# Latency Kills — one physical T4 lane

This notebook hosts exactly one Llama 3.1 8B V4 motor lane. Open three independent copies for the three-T4 experiment. Outputs are private and the bearer token expires with the runtime.


In [ ]:
%pip -q install "transformers==5.15.0" "bitsandbytes==0.50.1" "accelerate==1.14.0" "fastapi>=0.116,<1" "uvicorn>=0.35,<1" "httpx>=0.28,<1"

import base64, json, os, pathlib, shutil, urllib.parse, urllib.request

branch = urllib.parse.quote('experiment/colab-three-t4-lanes', safe='')
api_url = (
    'https://api.github.com/repos/RPG-478/latency-kills/contents/'
    f'colab/remote_lane_server.py?ref={branch}'
)
with urllib.request.urlopen(api_url, timeout=30) as response:
    payload = json.load(response)
pathlib.Path('/content/remote_lane_server.py').write_bytes(
    base64.b64decode(payload['content'])
)

cloudflared_path = pathlib.Path('/content/cloudflared')
with urllib.request.urlopen(
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    timeout=120,
) as response, cloudflared_path.open('wb') as destination:
    shutil.copyfileobj(response, destination)
cloudflared_path.chmod(0o700)
print('BOOTSTRAP_READY', flush=True)


In [ ]:
import json, os, pathlib, re, secrets, statistics, subprocess, sys, time
import requests
from google.colab import userdata

lane_name = f't4-{secrets.token_hex(3)}'
lane_token = secrets.token_urlsafe(32)
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Colab SecretsにHF_TOKENが見つかりません')

server_env = os.environ.copy()
server_env.update({
    'HF_TOKEN': hf_token,
    'LATENCY_KILLS_LANE_NAME': lane_name,
    'LATENCY_KILLS_LANE_TOKEN': lane_token,
    'LATENCY_KILLS_CONSTRAIN_DIGITS': '0',
})
server_log_path = pathlib.Path('/content/remote-lane-server.log')
server_log = server_log_path.open('w', encoding='utf-8')
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'remote_lane_server:app',
     '--host', '127.0.0.1', '--port', '8000', '--log-level', 'warning'],
    cwd='/content', env=server_env, stdout=server_log, stderr=subprocess.STDOUT,
)
auth_headers = {'Authorization': f'Bearer {lane_token}'}
health = None
for _ in range(900):
    if server_proc.poll() is not None:
        server_log.flush()
        raise RuntimeError(server_log_path.read_text(errors='replace')[-4000:])
    try:
        response = requests.get('http://127.0.0.1:8000/health', headers=auth_headers, timeout=1)
        if response.ok:
            health = response.json()
            break
    except requests.RequestException:
        pass
    time.sleep(1)
if health is None:
    raise RuntimeError('model server did not become ready within 15 minutes')

local_latencies = []
semantic_cases = [
    ('v=0 x=9999 a=10', '4'), ('v=1 x=0 a=0', '0'),
    ('v=1 x=-350 a=10', '2'), ('v=1 x=-150 a=10', '1'),
    ('v=1 x=0 a=10', '5'), ('v=1 x=150 a=10', '3'),
    ('v=1 x=350 a=10', '4'),
]
semantic_rows = []
for index, (observation, expected) in enumerate(semantic_cases):
    started = time.perf_counter()
    response = requests.post(
        'http://127.0.0.1:8000/motor', headers=auth_headers,
        json={'request_id': f'probe-{index}', 'observation': observation}, timeout=10,
    )
    response.raise_for_status()
    row = response.json()
    local_latencies.append((time.perf_counter() - started) * 1000)
    semantic_rows.append({
        'observation': observation, 'expected': expected, 'actual': row['token'],
        'correct': row['token'] == expected, 'compute_ms': row['compute_ms'],
    })
if not all(row['correct'] for row in semantic_rows):
    raise RuntimeError(f'semantic probe failed: {semantic_rows}')

tunnel_log_path = pathlib.Path('/content/cloudflared.log')
tunnel_log = tunnel_log_path.open('w', encoding='utf-8')
tunnel_proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000',
     '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
endpoint = None
for _ in range(120):
    if tunnel_proc.poll() is not None:
        tunnel_log.flush()
        raise RuntimeError(tunnel_log_path.read_text(errors='replace')[-4000:])
    tunnel_log.flush()
    match = re.search(
        r'https://[a-z0-9-]+\.trycloudflare\.com',
        tunnel_log_path.read_text(errors='replace'),
    )
    if match:
        endpoint = match.group(0)
        break
    time.sleep(0.5)
if endpoint is None:
    raise RuntimeError('Cloudflare Quick Tunnel URL was not reported')

public_health = requests.get(endpoint + '/health', headers=auth_headers, timeout=20)
public_health.raise_for_status()
lane_config = {'name': lane_name, 'endpoint': endpoint, 'token': lane_token}
print('LOCAL_SEMANTIC', json.dumps(semantic_rows, ensure_ascii=False), flush=True)
print('LOCAL_WIRE_MS', json.dumps({
    'mean': statistics.fmean(local_latencies),
    'p50': statistics.median(local_latencies),
    'max': max(local_latencies),
}), flush=True)
print('__LATENCY_KILLS_LANE__' + json.dumps(lane_config), flush=True)
print('LANE_READY; keep this cell running until the experiment is finished.', flush=True)
while server_proc.poll() is None and tunnel_proc.poll() is None:
    time.sleep(60)
raise RuntimeError('lane server or tunnel stopped unexpectedly')
